In [3]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().resolve()
while not (project_root / 'data' / 'raw').is_dir() and project_root != project_root.parent:
    project_root = project_root.parent

raw_data_dir = project_root / 'data' / 'raw'
processed_data_dir = project_root / 'data' / 'processed'
processed_data_dir.mkdir(parents=True, exist_ok=True)

# Load telemetry and shop lookup data
robot_1_df = pd.read_csv(raw_data_dir / 'robot_1.csv')
robot_2_df = pd.read_csv(raw_data_dir / 'robot_2.csv')
robot_3_df = pd.read_csv(raw_data_dir / 'robot_3.csv')
robot_shop_df = pd.read_csv(raw_data_dir / 'robot_shop.csv')

# Confirm shop_id is a primary key in the shop table
assert robot_shop_df['shop_id'].is_unique

# Confirm each robot's shop foreign keys resolve, then merge with shop details
robot_dataframes = [robot_1_df, robot_2_df, robot_3_df]
merged_robot_dataframes = []
for robot_df in robot_dataframes:
    assert robot_df['shop_id'].notna().all()
    assert robot_df['shop_id'].isin(robot_shop_df['shop_id']).all()
    merged_robot_dataframes.append(
        pd.merge(
            robot_df,
            robot_shop_df,
            on='shop_id',
            how='left',
            validate='many_to_one',
        )
    )

# Combine all robot/shop merges into one processed dataset
merged_df = pd.concat(merged_robot_dataframes, ignore_index=True)
expected_records = sum(len(robot_df) for robot_df in robot_dataframes)
assert len(merged_df) == expected_records
assert merged_df['Robot_ID'].nunique() == 3

output_path = processed_data_dir / 'electrical_current_with_shop.csv'
merged_df.to_csv(output_path, index=False)

print('Merged robot shapes:', [frame.shape for frame in merged_robot_dataframes])
print('Combined shape:', merged_df.shape)
print('Distinct robots:', merged_df['Robot_ID'].nunique())
print('Saved:', output_path)
print(merged_df.head())

Merged robot shapes: [(39672, 22), (10, 22), (10, 22)]
Combined shape: (39692, 22)
Distinct robots: 3
Saved: C:\Personal\Projects\wrangling-workshop\data\processed\electrical_current_with_shop.csv
     Trait  Axis #1  Axis #2  Axis #3  Axis #4  Axis #5  Axis #6  Axis #7  \
0  current      0.0      0.0      0.0      0.0      0.0      0.0      0.0   
1  current      0.0      0.0      0.0      0.0      0.0      0.0      0.0   
2  current      0.0      0.0      0.0      0.0      0.0      0.0      0.0   
3  current      0.0      0.0      0.0      0.0      0.0      0.0      0.0   
4  current      0.0      0.0      0.0      0.0      0.0      0.0      0.0   

   Axis #8  Axis #9  ...  Axis #12  Axis #13  Axis #14  \
0      0.0      NaN  ...       NaN       NaN       NaN   
1      0.0      NaN  ...       NaN       NaN       NaN   
2      0.0      NaN  ...       NaN       NaN       NaN   
3      0.0      NaN  ...       NaN       NaN       NaN   
4      0.0      NaN  ...       NaN       NaN      